In [22]:
# Use inline backend for Jupyter
%matplotlib inline

import os
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from IPython.display import display, clear_output
import ipywidgets as widgets

def interactive_threshold_adjuster(input_folder):
    """
    Interactive tool to manually adjust thresholds for each image in a folder.
    Uses ipywidgets for Jupyter notebook compatibility.
    Automatically saves binary images.
    """
    
    # Get all image files
    valid_extensions = ('.tiff', '.tif', '.png', '.jpg', '.jpeg')
    image_files = sorted([f for f in os.listdir(input_folder) 
                         if f.lower().endswith(valid_extensions)])
    
    if not image_files:
        print(f"No images found in {input_folder}")
        return
    
    print(f"Found {len(image_files)} images to process")
    
    # Create output folder
    thresholded_folder = os.path.join(input_folder, "Baseline_Thresholded_Images")
    os.makedirs(thresholded_folder, exist_ok=True)
    print(f"Binary images will be saved to: {thresholded_folder}")
    
    # Storage
    threshold_results = []
    current_state = {
        'index': 0,
        'image': None,
        'filename': None
    }
    
    def load_image(idx):
        if idx >= len(image_files):
            print("\n" + "="*80)
            print("All images processed!")
            print("="*80)
            
            df = pd.DataFrame(threshold_results)
            output_excel = os.path.join(input_folder, "Baseline_Threshold_Values.xlsx")
            df.to_excel(output_excel, index=False)
            print(f"\nThreshold values saved to: {output_excel}")
            print(f"Binary images saved to: {thresholded_folder}")
            return None, None
        
        file_name = image_files[idx]
        image_path = os.path.join(input_folder, file_name)
        image = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
        
        if image is None:
            print(f"Error loading {file_name}, skipping...")
            return load_image(idx + 1)
        
        return image, file_name
    
    def display_images(threshold_value):
        if current_state['image'] is None:
            return
        
        image = current_state['image']
        file_name = current_state['filename']
        binary = image > threshold_value
        
        # Create figure
        fig, axes = plt.subplots(2, 2, figsize=(11, 9))
        fig.suptitle(f"Image {current_state['index'] + 1} of {len(image_files)}: {file_name}", 
                    fontsize=10, fontweight='bold')
        
        # Original
        axes[0, 0].imshow(image, cmap='gray')
        axes[0, 0].set_title('Original Image', fontsize=9, fontweight='bold')
        axes[0, 0].axis('off')
        
        # Histogram
        hist, bins = np.histogram(image.flatten(), bins=256, range=[0, 256])
        axes[0, 1].plot(bins[:-1], hist, color='black', linewidth=1)
        axes[0, 1].axvline(threshold_value, color='red', linestyle='--', linewidth=2, 
                          label=f'Threshold: {threshold_value}')
        axes[0, 1].set_title('Histogram', fontsize=9, fontweight='bold')
        axes[0, 1].set_xlabel('Pixel Intensity', fontsize=8)
        axes[0, 1].set_ylabel('Frequency', fontsize=8)
        axes[0, 1].tick_params(labelsize=7)
        axes[0, 1].legend(fontsize=8)
        axes[0, 1].grid(True, alpha=0.3)
        
        # Thresholded
        axes[1, 0].imshow(binary, cmap='gray')
        axes[1, 0].set_title(f'Thresholded (> {threshold_value})', fontsize=9, fontweight='bold')
        axes[1, 0].axis('off')
        
        # Overlay
        from skimage.segmentation import find_boundaries
        overlay = cv2.cvtColor(image, cv2.COLOR_GRAY2RGB)
        boundaries = find_boundaries(binary, mode='outer')
        overlay[boundaries] = [255, 0, 0]
        
        axes[1, 1].imshow(overlay)
        axes[1, 1].set_title('Overlay (Red = Threshold)', fontsize=9, fontweight='bold')
        axes[1, 1].axis('off')
        
        plt.tight_layout()
        plt.show()
    
    # Create interactive widgets
    threshold_slider = widgets.IntSlider(
        value=127,
        min=0,
        max=255,
        step=1,
        description='Threshold:',
        style={'description_width': '100px'},
        layout=widgets.Layout(width='500px')
    )
    
    next_button = widgets.Button(
        description='Next Image',
        button_style='success',
        layout=widgets.Layout(width='150px')
    )
    
    status_label = widgets.Label(value='Ready')
    
    output_area = widgets.Output()
    
    def update_display(threshold_value):
        with output_area:
            clear_output(wait=True)
            display_images(threshold_value)
    
    def on_threshold_change(change):
        update_display(change['new'])
    
    def on_next_click(b):
        if current_state['image'] is None:
            return
        
        threshold_value = threshold_slider.value
        file_name = current_state['filename']
        
        # Save result
        threshold_results.append({
            'Image Name': file_name,
            'Threshold Value': threshold_value
        })
        
        # Automatically save binary image
        binary = current_state['image'] > threshold_value
        binary_uint8 = (binary * 255).astype(np.uint8)
        output_path = os.path.join(thresholded_folder, 
                                  file_name.replace('.tiff', '.png').replace('.tif', '.png'))
        cv2.imwrite(output_path, binary_uint8)
        
        print(f"[{current_state['index'] + 1}/{len(image_files)}] {file_name}: Threshold = {threshold_value} | Saved: {os.path.basename(output_path)}")
        
        # Move to next
        current_state['index'] += 1
        
        # Load next image
        next_image, next_file = load_image(current_state['index'])
        if next_image is not None:
            current_state['image'] = next_image
            current_state['filename'] = next_file
            
            # Temporarily disconnect to avoid triggering update twice
            threshold_slider.unobserve(on_threshold_change, names='value')
            threshold_slider.value = int(np.mean(next_image))
            threshold_slider.observe(on_threshold_change, names='value')
            
            update_display(threshold_slider.value)
            status_label.value = f"Image {current_state['index'] + 1} of {len(image_files)} | Last saved: {os.path.basename(output_path)}"
        else:
            status_label.value = "All images processed!"
    
    # Load first image
    image, file_name = load_image(0)
    if image is not None:
        current_state['image'] = image
        current_state['filename'] = file_name
        threshold_slider.value = int(np.mean(image))
    
    # Connect events
    threshold_slider.observe(on_threshold_change, names='value')
    next_button.on_click(on_next_click)
    
    # Layout - images on top, controls on bottom
    controls = widgets.VBox([
        widgets.HBox([threshold_slider, next_button]),
        status_label
    ])
    
    # Display output first, then controls below
    display(output_area)
    display(controls)
    
    # Initial display
    update_display(threshold_slider.value)

# Run it
interactive_threshold_adjuster('/Users/allisonpickle/Desktop/Neurons')

Found 57 images to process
Binary images will be saved to: /Users/allisonpickle/Desktop/Neurons/Baseline_Thresholded_Images


Output()

In [23]:
import os
import cv2
import numpy as np
from skimage.filters import threshold_otsu, threshold_li, threshold_yen, threshold_isodata, threshold_triangle, threshold_minimum

def generate_threshold_images(microglia_folder, output_folder):
    """
    Generate thresholded images using different methods.
    No statistics, just creates the images in labeled folders.
    
    Parameters:
    -----------
    microglia_folder : str
        Path to folder containing images to threshold
    output_folder : str
        Path to save thresholded images
    """
    
    # Get all images in microglia folder
    valid_extensions = ('.tiff', '.tif', '.png', '.jpg', '.jpeg')
    image_files = sorted([f for f in os.listdir(microglia_folder) 
                         if f.lower().endswith(valid_extensions)])
    
    if not image_files:
        print(f"No images found in {microglia_folder}")
        return
    
    print(f"Found {len(image_files)} images to process")
    
    # Create output folders for each method
    os.makedirs(output_folder, exist_ok=True)
    
    method_names = ['Mean', 'Otsu', 'Li', 'Yen', 'ISODATA', 'Triangle', 'Minimum']
    method_folders = {}
    
    for method_name in method_names:
        method_folder = os.path.join(output_folder, f"{method_name}_Thresholded")
        os.makedirs(method_folder, exist_ok=True)
        method_folders[method_name] = method_folder
        print(f"Created folder: {method_folder}")
    
    # Thresholding methods
    methods = {
        'Otsu': threshold_otsu,
        'Li': threshold_li,
        'Yen': threshold_yen,
        'ISODATA': threshold_isodata,
        'Triangle': threshold_triangle,
        'Minimum': threshold_minimum
    }
    
    # Process each image
    for img_file in image_files:
        print(f"\nProcessing: {img_file}")
        
        # Load image
        img_path = os.path.join(microglia_folder, img_file)
        image = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        
        if image is None:
            print(f"  Error loading {img_file}, skipping...")
            continue
        
        # Output filename (convert to PNG)
        output_filename = img_file.replace('.tiff', '.png').replace('.tif', '.png')
        
        # Mean threshold
        mean_threshold = int(np.mean(image))
        mean_binary = (image > mean_threshold).astype(np.uint8) * 255
        mean_output = os.path.join(method_folders['Mean'], output_filename)
        cv2.imwrite(mean_output, mean_binary)
        print(f"  ✓ Mean (threshold: {mean_threshold})")
        
        # Apply each automatic thresholding method
        for method_name, method_func in methods.items():
            try:
                threshold_value = method_func(image)
                binary = (image > threshold_value).astype(np.uint8) * 255
                method_output = os.path.join(method_folders[method_name], output_filename)
                cv2.imwrite(method_output, binary)
                print(f"  ✓ {method_name} (threshold: {int(threshold_value)})")
            except Exception as e:
                print(f"  ✗ {method_name}: Error - {str(e)}")
    
    print("\n" + "="*80)
    print("PROCESSING COMPLETE!")
    print("="*80)
    print(f"Output folder: {output_folder}")
    print(f"\nGenerated folders:")
    for method_name in method_names:
        print(f"  • {method_name}_Thresholded/")

# Example usage:
microglia_folder = '/Users/allisonpickle/Desktop/Neurons'
output_folder = '/Users/allisonpickle/Desktop/Neurons/Threshold_Results'

generate_threshold_images(microglia_folder, output_folder)

Found 57 images to process
Created folder: /Users/allisonpickle/Desktop/Neurons/Threshold_Results/Mean_Thresholded
Created folder: /Users/allisonpickle/Desktop/Neurons/Threshold_Results/Otsu_Thresholded
Created folder: /Users/allisonpickle/Desktop/Neurons/Threshold_Results/Li_Thresholded
Created folder: /Users/allisonpickle/Desktop/Neurons/Threshold_Results/Yen_Thresholded
Created folder: /Users/allisonpickle/Desktop/Neurons/Threshold_Results/ISODATA_Thresholded
Created folder: /Users/allisonpickle/Desktop/Neurons/Threshold_Results/Triangle_Thresholded
Created folder: /Users/allisonpickle/Desktop/Neurons/Threshold_Results/Minimum_Thresholded

Processing: 240727_Tri_Ctrl_11_A_10_C_2.tiff
  ✓ Mean (threshold: 16)
  ✓ Otsu (threshold: 32)
  ✓ Li (threshold: 16)
  ✓ Yen (threshold: 107)
  ✓ ISODATA (threshold: 30)
  ✓ Triangle (threshold: 16)
  ✓ Minimum (threshold: 210)

Processing: 240727_Tri_Ctrl_12_A_10_C_2.tiff
  ✓ Mean (threshold: 13)
  ✓ Otsu (threshold: 24)
  ✓ Li (threshold: 13)
 

In [ ]:
import os
import cv2
import numpy as np
import pandas as pd

def calculate_threshold_accuracy(baseline_folder, threshold_results_folder, output_excel=None):
    """
    Compare thresholded images against baseline ground truth.
    Calculates pixel-wise TP, FN, FP, TN and specified metrics for each method.
    
    Parameters:
    -----------
    baseline_folder : str
        Path to folder containing baseline (ground truth) binary images
    threshold_results_folder : str
        Path to folder containing method subfolders (e.g., Otsu_Thresholded/)
    output_excel : str, optional
        Path to save results Excel file
    
    Returns:
    --------
    DataFrame with accuracy metrics for each image and method
    """
    
    # Get all baseline images
    valid_extensions = ('.tiff', '.tif', '.png', '.jpg', '.jpeg')
    baseline_files = sorted([f for f in os.listdir(baseline_folder) 
                            if f.lower().endswith(valid_extensions)])
    
    if not baseline_files:
        print(f"No baseline images found in {baseline_folder}")
        return None, None
    
    print(f"Found {len(baseline_files)} baseline images")
    
    # Find all method folders
    method_folders = {}
    for item in os.listdir(threshold_results_folder):
        item_path = os.path.join(threshold_results_folder, item)
        if os.path.isdir(item_path) and '_Thresholded' in item:
            method_name = item.replace('_Thresholded', '')
            method_folders[method_name] = item_path
    
    if not method_folders:
        print(f"No method folders found in {threshold_results_folder}")
        return None, None
    
    print(f"Found {len(method_folders)} thresholding methods: {list(method_folders.keys())}")
    
    # Storage for results
    results = []
    
    # Process each baseline image
    for baseline_file in baseline_files:
        print(f"\nProcessing: {baseline_file}")
        
        # Load baseline (ground truth)
        baseline_path = os.path.join(baseline_folder, baseline_file)
        baseline_img = cv2.imread(baseline_path, cv2.IMREAD_GRAYSCALE)
        
        if baseline_img is None:
            print(f"  Error loading baseline {baseline_file}, skipping...")
            continue
        
        # Convert to binary (0 or 1)
        baseline_binary = (baseline_img > 127).astype(np.uint8)
        
        # Compare against each method
        for method_name, method_folder in method_folders.items():
            # Find corresponding predicted image
            predicted_path = os.path.join(method_folder, baseline_file)
            
            # Try alternate extensions if exact match not found
            if not os.path.exists(predicted_path):
                # Try .png version
                predicted_path = os.path.join(method_folder, 
                                             baseline_file.replace('.tiff', '.png').replace('.tif', '.png'))
            
            if not os.path.exists(predicted_path):
                print(f"  Warning: {method_name} - No matching image found")
                continue
            
            # Load predicted image
            predicted_img = cv2.imread(predicted_path, cv2.IMREAD_GRAYSCALE)
            
            if predicted_img is None:
                print(f"  Error loading {method_name} prediction")
                continue
            
            # Convert to binary (0 or 1)
            predicted_binary = (predicted_img > 127).astype(np.uint8)
            
            # Ensure same dimensions
            if baseline_binary.shape != predicted_binary.shape:
                print(f"  Warning: {method_name} - Shape mismatch, resizing...")
                predicted_binary = cv2.resize(predicted_binary, 
                                             (baseline_binary.shape[1], baseline_binary.shape[0]),
                                             interpolation=cv2.INTER_NEAREST)
            
            # Calculate confusion matrix
            # True Positive: baseline=1, predicted=1
            TP = np.sum((baseline_binary == 1) & (predicted_binary == 1))
            
            # False Negative: baseline=1, predicted=0
            FN = np.sum((baseline_binary == 1) & (predicted_binary == 0))
            
            # False Positive: baseline=0, predicted=1
            FP = np.sum((baseline_binary == 0) & (predicted_binary == 1))
            
            # True Negative: baseline=0, predicted=0
            TN = np.sum((baseline_binary == 0) & (predicted_binary == 0))
            
            # Calculate requested metrics
            # Dice Coefficient: 2*TP / (2*TP + FP + FN)
            dice = (2 * TP) / (2 * TP + FP + FN) if (2 * TP + FP + FN) > 0 else 0
            
            # Jaccard Index: TP / (TP + FP + FN)
            jaccard = TP / (TP + FP + FN) if (TP + FP + FN) > 0 else 0
            
            # Precision: TP / (TP + FP)
            precision = TP / (TP + FP) if (TP + FP) > 0 else 0
            
            # Recall: TP / (TP + FN)
            recall = TP / (TP + FN) if (TP + FN) > 0 else 0
            
            # Store results
            result = {
                'Image Name': baseline_file,
                'Method': method_name,
                'True Positive (TP)': int(TP),
                'False Negative (FN)': int(FN),
                'False Positive (FP)': int(FP),
                'True Negative (TN)': int(TN),
                'Dice Coefficient': dice,
                'Jaccard Index': jaccard,
                'Precision': precision,
                'Recall': recall
            }
            
            results.append(result)
            
            print(f"  {method_name}:")
            print(f"    TP={TP:,}, FN={FN:,}, FP={FP:,}, TN={TN:,}")
            print(f"    Dice={dice:.4f}, Jaccard={jaccard:.4f}, Precision={precision:.4f}, Recall={recall:.4f}")
    
    # Create results DataFrame
    results_df = pd.DataFrame(results)
    
    if results_df.empty:
        print("\nNo results generated!")
        return None, None
    
    # Calculate average metrics per method
    print("\n" + "="*80)
    print("AVERAGE METRICS BY THRESHOLD TYPE")
    print("="*80)
    
    summary_data = []
    for method in results_df['Method'].unique():
        method_data = results_df[results_df['Method'] == method]
        
        summary = {
            'Method': method,
            'Num Images': len(method_data),
            'Average Dice Coefficient': method_data['Dice Coefficient'].mean(),
            'Average Jaccard Index': method_data['Jaccard Index'].mean(),
            'Average Precision': method_data['Precision'].mean(),
            'Average Recall': method_data['Recall'].mean()
        }
        
        summary_data.append(summary)
        
        print(f"\n{method}:")
        print(f"  Images processed: {summary['Num Images']}")
        print(f"  Average Dice Coefficient: {summary['Average Dice Coefficient']:.4f}")
        print(f"  Average Jaccard Index: {summary['Average Jaccard Index']:.4f}")
        print(f"  Average Precision: {summary['Average Precision']:.4f}")
        print(f"  Average Recall: {summary['Average Recall']:.4f}")
    
    summary_df = pd.DataFrame(summary_data)
    
    # Save to Excel if output path specified
    if output_excel:
        with pd.ExcelWriter(output_excel, engine='openpyxl') as writer:
            # Image-level results
            results_df.to_excel(writer, sheet_name='Image-Level Results', index=False)
            
            # Average metrics by method
            summary_df.to_excel(writer, sheet_name='Average by Method', index=False)
        
        print(f"\n{'='*80}")
        print(f"Results saved to: {output_excel}")
        print(f"{'='*80}")
    
    return results_df, summary_df

# Example usage:
baseline_folder = '/Users/allisonpickle/Desktop/Threshold Comparison Table/Microglia_To_Threshold'
threshold_results_folder = '/Users/allisonpickle/Desktop/Threshold Comparison Table/Isolated microglia'
output_excel = '/Users/allisonpickle/Desktop/Threshold_Accuracy_Analysis.xlsx'

results_df, summary_df = calculate_threshold_accuracy(baseline_folder, threshold_results_folder, output_excel)

# Display results
if results_df is not None and summary_df is not None:
    print("\n" + "="*80)
    print("SUMMARY TABLE")
    print("="*80)
    display(summary_df)
else:
    print("\n❌ Error: Check that baseline folder exists and contains images")
    print(f"Baseline folder: {baseline_folder}")
    print(f"Threshold results folder: {threshold_results_folder}")

Found 57 baseline images
Found 7 thresholding methods: ['Otsu', 'Minimum', 'Yen', 'Li', 'Triangle', 'Mean', 'ISODATA']

Processing: 240727_Tri_Ctrl_11_A_10_C_2.png
  Otsu:
    TP=167,033, FN=382,145, FP=0, TN=894,342
    Dice=0.4664, Jaccard=0.3042, Precision=1.0000, Recall=0.3042
  Minimum:
    TP=907, FN=548,271, FP=0, TN=894,342
    Dice=0.0033, Jaccard=0.0017, Precision=1.0000, Recall=0.0017
  Yen:
    TP=7,154, FN=542,024, FP=0, TN=894,342
    Dice=0.0257, Jaccard=0.0130, Precision=1.0000, Recall=0.0130
  Li:
    TP=509,326, FN=39,852, FP=0, TN=894,342
    Dice=0.9624, Jaccard=0.9274, Precision=1.0000, Recall=0.9274
  Triangle:
    TP=509,326, FN=39,852, FP=0, TN=894,342
    Dice=0.9624, Jaccard=0.9274, Precision=1.0000, Recall=0.9274
  Mean:
    TP=509,326, FN=39,852, FP=0, TN=894,342
    Dice=0.9624, Jaccard=0.9274, Precision=1.0000, Recall=0.9274
  ISODATA:
    TP=189,710, FN=359,468, FP=0, TN=894,342
    Dice=0.5135, Jaccard=0.3454, Precision=1.0000, Recall=0.3454

Processing:

,Method,Num Images,Average Dice Coefficient,Average Jaccard Index,Average Precision,Average Recall
0,Otsu,57,0.812804,0.712578,0.913311,0.799268
1,Minimum,54,0.003863,0.001945,1.000000,0.001945
2,Yen,57,0.025962,0.013520,1.000000,0.013520
3,Li,57,0.761190,0.634989,0.650058,0.984931
4,Triangle,57,0.767429,0.651746,0.705289,0.946457
5,Mean,57,0.763548,0.639849,0.643291,0.996558
6,ISODATA,57,0.814068,0.711766,0.878823,0.832943


In [26]:
import os
import cv2
import numpy as np
from PIL import Image
import pandas as pd

def compile_threshold_comparison_grid(baseline_folder, threshold_results_folder, output_folder=None, spacing=10):
    """
    Create side-by-side comparison images showing baseline and all threshold methods.
    Each row shows: Original | Otsu | Li | Yen | ISODATA | Triangle | Mean | Minimum Error
    
    Parameters:
    -----------
    baseline_folder : str
        Path to folder containing baseline (ground truth) binary images
    threshold_results_folder : str
        Path to folder containing method subfolders (e.g., Otsu_Thresholded/)
    output_folder : str, optional
        Path to save compiled comparison images
    spacing : int, optional
        Spacing between images in pixels (default: 10)
    
    Returns:
    --------
    List of successfully compiled image names
    """
    
    # Define method order (as requested)
    method_order = ['Otsu', 'Li', 'Yen', 'ISODATA', 'Triangle', 'Mean', 'Minimum_Error']
    
    # Get all baseline images
    valid_extensions = ('.tiff', '.tif', '.png', '.jpg', '.jpeg')
    baseline_files = sorted([f for f in os.listdir(baseline_folder) 
                            if f.lower().endswith(valid_extensions)])
    
    if not baseline_files:
        print(f"No baseline images found in {baseline_folder}")
        return []
    
    print(f"Found {len(baseline_files)} baseline images")
    
    # Map method names to folder paths
    method_folders = {}
    for method in method_order:
        folder_name = f"{method}_Thresholded"
        folder_path = os.path.join(threshold_results_folder, folder_name)
        if os.path.exists(folder_path):
            method_folders[method] = folder_path
            print(f"✓ Found: {method}")
        else:
            print(f"⚠️  Missing: {method} (folder: {folder_name})")
    
    if not method_folders:
        print("No threshold method folders found!")
        return []
    
    # Create output folder if specified
    if output_folder:
        os.makedirs(output_folder, exist_ok=True)
        print(f"\nOutput folder: {output_folder}")
    
    successfully_compiled = []
    
    # Process each baseline image
    for idx, baseline_file in enumerate(baseline_files, 1):
        print(f"\n[{idx}/{len(baseline_files)}] Processing: {baseline_file}")
        
        # Load baseline image
        baseline_path = os.path.join(baseline_folder, baseline_file)
        baseline_img = cv2.imread(baseline_path, cv2.IMREAD_GRAYSCALE)
        
        if baseline_img is None:
            print(f"  ❌ Error loading baseline image, skipping...")
            continue
        
        # Storage for images in order: [Original, Otsu, Li, Yen, ISODATA, Triangle, Mean, Minimum Error]
        images_to_compile = [baseline_img]
        labels = ['Original']
        
        # Load each threshold method in specified order
        for method in method_order:
            if method not in method_folders:
                print(f"  ⚠️  Skipping {method} (folder not found)")
                continue
            
            method_folder = method_folders[method]
            
            # Try to find matching file
            predicted_path = os.path.join(method_folder, baseline_file)
            
            # Try alternate extensions
            if not os.path.exists(predicted_path):
                predicted_path = os.path.join(method_folder, 
                                             baseline_file.replace('.tiff', '.png').replace('.tif', '.png'))
            
            if not os.path.exists(predicted_path):
                print(f"  ⚠️  {method}: No matching image found")
                # Use blank placeholder
                images_to_compile.append(np.zeros_like(baseline_img))
                labels.append(f"{method} (Missing)")
                continue
            
            # Load threshold result
            threshold_img = cv2.imread(predicted_path, cv2.IMREAD_GRAYSCALE)
            
            if threshold_img is None:
                print(f"  ❌ {method}: Error loading image")
                images_to_compile.append(np.zeros_like(baseline_img))
                labels.append(f"{method} (Error)")
                continue
            
            # Resize if dimensions don't match
            if threshold_img.shape != baseline_img.shape:
                print(f"  🔧 {method}: Resizing from {threshold_img.shape} to {baseline_img.shape}")
                threshold_img = cv2.resize(threshold_img, 
                                          (baseline_img.shape[1], baseline_img.shape[0]),
                                          interpolation=cv2.INTER_NEAREST)
            
            images_to_compile.append(threshold_img)
            labels.append(method)
            print(f"  ✓ {method}: Loaded")
        
        # Create side-by-side compilation with spacing
        # Add labels above each image
        label_height = 30
        img_height, img_width = baseline_img.shape
        
        # Create labeled images
        labeled_images = []
        for img, label in zip(images_to_compile, labels):
            # Create white background for label
            label_img = np.ones((label_height, img_width), dtype=np.uint8) * 255
            
            # Add text label
            font = cv2.FONT_HERSHEY_SIMPLEX
            font_scale = 0.5
            thickness = 1
            text_size = cv2.getTextSize(label, font, font_scale, thickness)[0]
            text_x = (img_width - text_size[0]) // 2
            text_y = (label_height + text_size[1]) // 2
            cv2.putText(label_img, label, (text_x, text_y), font, font_scale, 0, thickness)
            
            # Stack label on top of image
            labeled_img = np.vstack([label_img, img])
            labeled_images.append(labeled_img)
        
        # Add spacing between images
        spaced_images = []
        for i, labeled_img in enumerate(labeled_images):
            spaced_images.append(labeled_img)
            # Add white spacer between images (except after last image)
            if i < len(labeled_images) - 1:
                spacer = np.ones((labeled_img.shape[0], spacing), dtype=np.uint8) * 255
                spaced_images.append(spacer)
        
        # Concatenate horizontally
        compiled_image = np.hstack(spaced_images)
        
        # Save compiled image if output folder specified
        if output_folder:
            output_filename = f"Comparison_{baseline_file}"
            output_path = os.path.join(output_folder, output_filename)
            
            # Save as PNG for better quality
            output_path = output_path.replace('.tiff', '.png').replace('.tif', '.png')
            cv2.imwrite(output_path, compiled_image)
            print(f"  💾 Saved: {output_filename}")
        
        successfully_compiled.append(baseline_file)
    
    # Summary
    print(f"\n{'='*80}")
    print(f"COMPILATION COMPLETE")
    print(f"{'='*80}")
    print(f"Successfully compiled: {len(successfully_compiled)}/{len(baseline_files)} images")
    
    if output_folder:
        print(f"Comparison images saved to: {output_folder}")
    
    return successfully_compiled

# Example usage:
baseline_folder = '/Users/allisonpickle/Desktop/Neurons/Baseline_Thresholded_Images'
threshold_results_folder = '/Users/allisonpickle/Desktop/Neurons/Threshold_Results'
output_folder = '/Users/allisonpickle/Desktop/Neurons/Compiled_Comparisons'

compiled_images = compile_threshold_comparison_grid(
    baseline_folder, 
    threshold_results_folder, 
    output_folder,
    spacing=10  # Adjust this value to change spacing (pixels)
)

print(f"\n✅ Compiled {len(compiled_images)} comparison images")

Found 57 baseline images
✓ Found: Otsu
✓ Found: Li
✓ Found: Yen
✓ Found: ISODATA
✓ Found: Triangle
✓ Found: Mean
✓ Found: Minimum_Error

Output folder: /Users/allisonpickle/Desktop/Neurons/Compiled_Comparisons

[1/57] Processing: 240727_Tri_Ctrl_11_A_10_C_2.png
  ✓ Otsu: Loaded
  ✓ Li: Loaded
  ✓ Yen: Loaded
  ✓ ISODATA: Loaded
  ✓ Triangle: Loaded
  ✓ Mean: Loaded
  ✓ Minimum_Error: Loaded
  💾 Saved: Comparison_240727_Tri_Ctrl_11_A_10_C_2.png

[2/57] Processing: 240727_Tri_Ctrl_12_A_10_C_2.png
  ✓ Otsu: Loaded
  ✓ Li: Loaded
  ✓ Yen: Loaded
  ✓ ISODATA: Loaded
  ✓ Triangle: Loaded
  ✓ Mean: Loaded
  ✓ Minimum_Error: Loaded
  💾 Saved: Comparison_240727_Tri_Ctrl_12_A_10_C_2.png

[3/57] Processing: 240727_Tri_Ctrl_14_A_10_C_2.png
  ✓ Otsu: Loaded
  ✓ Li: Loaded
  ✓ Yen: Loaded
  ✓ ISODATA: Loaded
  ✓ Triangle: Loaded
  ✓ Mean: Loaded
  ✓ Minimum_Error: Loaded
  💾 Saved: Comparison_240727_Tri_Ctrl_14_A_10_C_2.png

[4/57] Processing: 240727_Tri_Ctrl_16_A_10_C_2.png
  ✓ Otsu: Loaded
  ✓ Li: